In [ ]:
!pip install openai-whisper soundfile numpy torch python-multipart pyngrok nest-asyncio fastapi uvicorn
!apt-get install ffmpeg -y
# Replace with your actual ngrok token if needed
!ngrok config add-authtoken 3Ff1BERH5y1ApUfytKIzYacu5ke_ns5oJctGpiasmubg1pRz
!pip install httpx

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
import whisper
from google.colab import output
from IPython.display import HTML, display
from base64 import b64decode

# A robust HTML/JS interface that shows status and forces browser microphone access
RECORD_HTML = """
<div id="audio-recorder" style="padding: 10px; border: 1px solid #ccc; background-color: #f9f9f9; width: fit-content; border-radius: 5px;">
  <p id="status-text" style="font-weight: bold; color: #d9534f; margin: 0 0 10px 0;">🔴 Waiting for microphone permission...</p>
  <button id="stop-btn" style="padding: 8px 15px; background-color: #0275d8; color: white; border: none; border-radius: 4px; cursor: pointer;" disabled>Stop Recording</button>
</div>

<script>
(async () => {
  const status = document.getElementById('status-text');
  const btn = document.getElementById('stop-btn');

  try {
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const chunks = [];

    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.onstop = async () => {
      const blob = new Blob(chunks, { type: 'audio/wav' });
      const reader = new FileReader();
      reader.onloadend = () => {
        window.audioData = reader.result;
        status.innerText = "✅ Recording captured successfully!";
        status.style.color = "#5cb85c";
        btn.remove();
      };
      reader.readAsDataURL(blob);
      stream.getTracks().forEach(track => track.stop());
    };

    recorder.start();
    status.innerText = "🎤 Recording... Speak now!";
    status.style.color = "#f0ad4e";
    btn.disabled = false;

    // Auto stop after 6 seconds, or manually when clicking the button
    const autoStop = setTimeout(() => { if(recorder.state === "recording") recorder.stop(); }, 6000);

    btn.onclick = () => {
      clearTimeout(autoStop);
      if(recorder.state === "recording") recorder.stop();
    };

  } catch (err) {
    status.innerText = "❌ Error: Microphone access denied or unsupported!";
    status.style.color = "#d9534f";
    window.audioData = "ERROR";
  }
})();
</script>
"""

def record_live_audio(filename="live_input.wav"):
    # Clear out any previous data in the window variable
    output.eval_js("window.audioData = null;")

    # Display the recording widget
    display(HTML(RECORD_HTML))

    # Wait continuously until the JavaScript finishes recording and assigns window.audioData
    print("Waiting for browser recording process...")
    while True:
        data = output.eval_js("window.audioData")
        if data == "ERROR":
            raise RuntimeError("Microphone access was denied in your browser settings.")
        if data is not None:
            break

    # Decode and save the audio data
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    print("🛑 Saved to disk.")

# 1. Start the recording flow
record_live_audio("live_input.wav")

# 2. Load the Whisper model
print("\nLoading Whisper model...")
model = whisper.load_model("base")

# 3. Transcribe your recorded voice
print("Transcribing your speech...")
result = model.transcribe("live_input.wav")

print("\n--- Live Transcription Results ---")
print("Text:", result["text"])
print("Language detected:", result["language"])

Waiting for browser recording process...
🛑 Saved to disk.

Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 168MiB/s]


Transcribing your speech...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



--- Live Transcription Results ---
Text:  hello my
Language detected: en


In [ ]:
%%writefile stt_service.py
import whisper
import os
import uuid
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("Loading Whisper model ('base')...")
stt_model = whisper.load_model("base")
print("Whisper ready.")

@app.get("/")
def read_root():
    return {"message": "STT Service is running!"}

@app.post("/speech-to-text")
async def speech_to_text(audio: UploadFile = File(...)):
    temp_filename = f"temp_{uuid.uuid4()}.wav"

    with open(temp_filename, "wb") as f:
        content = await audio.read()
        f.write(content)

    try:
        result = stt_model.transcribe(temp_filename)
        return JSONResponse({
            "success": True,
            "text": result["text"].strip(),
            "language": result["language"]
        })
    except Exception as e:
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

Writing stt_service.py


In [ ]:
import asyncio
import nest_asyncio
import uvicorn
from pyngrok import ngrok
from stt_service import app

nest_asyncio.apply()
ngrok.kill()

public_url = ngrok.connect(8001)
print("\n" + "="*50)
print(f"🚀 PUBLIC API URL: {public_url}")
print("==================================================\n")

config = uvicorn.Config(app, host="0.0.0.0", port=8001, log_level="info")
server = uvicorn.Server(config)

loop = asyncio.get_event_loop()
loop.create_task(server.serve())

Loading Whisper model ('base')...
Whisper ready.

🚀 PUBLIC API URL: NgrokTunnel: "https://succulent-spotting-imbecile.ngrok-free.dev" -> "http://localhost:8001"



<Task pending name='Task-1' coro=<Server.serve() running at /usr/local/lib/python3.12/dist-packages/uvicorn/server.py:77>>

In [ ]:
import httpx
import asyncio
from google.colab import output
from base64 import b64decode

# JavaScript helper function to record audio in Google Colab
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = e => resolve(e.srcElement.result);
  reader.readAsDataURL(blob);
});
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  recorder = new MediaRecorder(stream);
  chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.onstop = async () => {
    blob = new Blob(chunks, { type: 'audio/wav' });
    res = await b2text(blob);
    resolve(res);
  };
  recorder.stop();
});
"""

def record_live_audio(filename="api_live_input.wav", seconds=5):
    print(f"🎤 Mic Active: Speak now for {seconds} seconds...")
    output.eval_js(RECORD_JS)
    data = output.eval_js(f'record({seconds} * 1000)')
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    print("🛑 Recording finished and saved to disk.")

# Define an async function to handle the request without deadlocking the server
async def send_audio_to_server():
    # 1. Record your live voice dynamically
    record_live_audio("api_live_input.wav", seconds=5)

    # 2. Set up API endpoint URL
    api_url = "http://localhost:8001/speech-to-text"

    # 3. POST the file asynchronously
    print("Sending dynamic mic input to your FastAPI server...")
    async with httpx.AsyncClient(timeout=30.0) as client:
        with open("api_live_input.wav", "rb") as audio_file:
            files = {"audio": ("api_live_input.wav", audio_file, "audio/wav")}
            response = await client.post(api_url, files=files)

    # 4. Print server response
    print("\n--- Server Response ---")
    print(response.json())

# Run the async loop inside Colab
await send_audio_to_server()

🎤 Mic Active: Speak now for 5 seconds...
🛑 Recording finished and saved to disk.
Sending dynamic mic input to your FastAPI server...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


INFO:     127.0.0.1:35990 - "POST /speech-to-text HTTP/1.1" 200 OK

--- Server Response ---
{'success': True, 'text': 'Hello, my name is Lavandale.', 'language': 'en'}


In [12]:
import httpx
import asyncio
import whisper
from google.colab import output
from base64 import b64decode
from IPython.display import HTML, display

# Upgraded HTML/JS Interface with explicit permission guidance
RECORD_HTML = """
<div id="audio-recorder" style="padding: 15px; border: 2px dashed #0275d8; background-color: #f8f9fa; width: 400px; border-radius: 8px; font-family: sans-serif; text-align: center;">
  <p id="status-text" style="font-weight: bold; color: #d9534f; margin: 0 0 12px 0;">🔒 Requesting Microphone Access...</p>
  <button id="stop-btn" style="padding: 10px 20px; background-color: #d9534f; color: white; border: none; border-radius: 5px; cursor: pointer; font-size: 14px; display: none;">Stop & Transcribe</button>
  <div id="help-msg" style="font-size: 12px; color: #555; margin-top: 5px; display: none;">
     Look at your browser's address bar (top left/right). Click the <b>Camera/Microphone icon</b> or <b>Lock icon</b> and select "Allow".
  </div>
</div>

<script>
(async () => {
  const status = document.getElementById('status-text');
  const btn = document.getElementById('stop-btn');
  const help = document.getElementById('help-msg');

  try {
    // Force prompt browser for permission
    const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
    const recorder = new MediaRecorder(stream);
    const chunks = [];

    recorder.ondataavailable = e => chunks.push(e.data);
    recorder.onstop = async () => {
      const blob = new Blob(chunks, { type: 'audio/wav' });
      const reader = new FileReader();
      reader.onloadend = () => {
        window.audioData = reader.result;
        status.innerText = " Audio captured successfully!";
        status.style.color = "#5cb85c";
        btn.remove();
        help.remove();
      };
      reader.readAsDataURL(blob);
      stream.getTracks().forEach(track => track.stop());
    };

    recorder.start();
    status.innerText = " Recording Active! Speak in Hindi or English now...";
    status.style.color = "#0275d8";
    btn.style.display = "inline-block";
    help.style.display = "none";

    // Auto-stop after 8 seconds max
    const autoStop = setTimeout(() => { if(recorder.state === "recording") recorder.stop(); }, 8000);

    btn.onclick = () => {
      clearTimeout(autoStop);
      if(recorder.state === "recording") recorder.stop();
    };

  } catch (err) {
    status.innerText = " Microphone access blocked by browser!";
    status.style.color = "#d9534f";
    help.style.display = "block";
    window.audioData = "ERROR";
  }
})();
</script>
"""

def get_live_audio(filename="multilingual_input.wav"):
    output.eval_js("window.audioData = null;")
    display(HTML(RECORD_HTML))

    while True:
        data = output.eval_js("window.audioData")
        if data == "ERROR":
            raise RuntimeError("\n ACTION REQUIRED: Click the Lock icon  or Microphone icon next to your browser's URL bar, change Microphone permission to 'Allow', then re-run this cell.")
        if data is not None:
            break

    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    print(" Recording saved successfully.")

async def run_multilingual_stt():
    try:
        get_live_audio("multilingual_input.wav")
    except RuntimeError as e:
        print(e)
        return

    print("\n Loading Multilingual Whisper model...")
    model = whisper.load_model("base")

    print(" Analyzing speech and transcribing...")
    result = model.transcribe("multilingual_input.wav", task="transcribe")

    print("\n" + "="*40)
    print(" --- MULTILINGUAL RESULTS --- ✨")
    print(f"Detected Language: {result.get('language', '').upper()}")
    print(f"Transcription   : {result['text'].strip()}")
    print("="*40)

await run_multilingual_stt()

 Recording saved successfully.

 Loading Multilingual Whisper model...
 Analyzing speech and transcribing...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



 --- MULTILINGUAL RESULTS --- ✨
Detected Language: HI
Transcription   : Namaste, My Name is Lavanya.
